# Naive 2D convolution

In [1]:
!nvidia-smi

Fri Jan 23 16:53:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile utils.h
/**
 * @file utils.h
 * @brief Utility functions for CUDA development
 *
 * This header provides core utility functions focusing on error checking timing,
 * initialising arrays in memory and result comparison.
 */

#ifndef UTILS_H
#define UTILS_H

#include <iostream>
#include <chrono>
#include <cstdlib>
#include <ctime>
#include <cmath>
#include <cuda_runtime.h>

/**
 * @brief CUDA error checking macro
 *
 * Evaluates a CUDA runtime call and checks for errors.
 * If an error is detected, prints detailed information and terminates the program.
 */
#define CUDA_CHECK(call) \
do { \
    cudaError_t error = call; \
    if (error != cudaSuccess) { \
        std::cerr << "CUDA error at " << __FILE__ << ":" << __LINE__ << " - " \
        << cudaGetErrorString(error) << " (" << error << ") " << std::endl; \
        exit(EXIT_FAILURE); \
} \
}while(0)

/**
 * @brief Initialise multiple arrays with random values in a specified range
 *
 * @param arrays     Array of pointers to initialize
 * @param num_arrays Number of arrays to initialize
 * @param size       Number of elements in each array
 * @param min       Minimum value for random numbers (default: 0.0)
 * @param max       Maximum value for random numbers (default: 1.0)
 * @param seed       Seed for random generator, 0 means use time(0) (default: 0)
 */
void initialiseArrays(float** arrays, int num_arrays, size_t size, float min=0.0f, float max=1.0f, unsigned int seed=0)
{
    // Set random seed
    if (seed == 0)
    {
        seed = static_cast<unsigned int>(time(0)); // get current time
    }
    srand(seed);

    float range = max - min;

    for (int i = 0; i < num_arrays; i++) // Iterate through each array pointer
    {
        for (size_t j = 0; j < size; j++) // Iterate through each element
        {
            arrays[i][j] = min + (static_cast<float>(rand()) / RAND_MAX) * range;
        }
    }
}

/**
 * @brief Measure CPU execution time using std::chrono
 *
 * @tparam Func     Function type
 * @param function  Function or lambda to measure
 * @return double   Execution time in milliseconds
 */
template <typename Function>
double measureExecutionTime(Function function)
{
    auto start = std::chrono::steady_clock::now();
    function();
    auto end = std::chrono::steady_clock::now();
    std::chrono::duration<double, std::milli> duration = (end - start);
    return duration.count();
}

/**
 * @brief Measure GPU kernel execution time using CUDA events
 *
 * @tparam KernelFunc  Kernel function type
 * @tparam Args        Kernel argument types
 * @param kernel       Kernel function to measure
 * @param grid         Grid dimensions
 * @param block        Block dimensions
 * @param args         Kernel arguments
 * @return float       Execution time in milliseconds
 */
template <typename KernelFunc>
float measureKernelTime(KernelFunc kernel)
{
    cudaEvent_t start;
    cudaEvent_t stop;
    float elapsed_time;

    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    // Start stopwatch
    CUDA_CHECK(cudaEventRecord(start));
    // Launch kernel
    kernel();
    // Stop stopwatch
    CUDA_CHECK(cudaEventRecord(stop));

    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&elapsed_time, start, stop));

    // Free events
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    return elapsed_time;
}

/**
 * @brief Compare results between CPU and GPU outputs using absolute and relative tolerance
 *
 * @param cpu_result  CPU computed results
 * @param gpu_result  GPU computed results
 * @param size        Number of elements to compare
 * @param atol        Absolute tolerance (default: 1e-4)
 * @param rtol        Relative tolerance (default: 1e-5)
 * @return bool       True if results match within tolerances, false otherwise
 */
bool compareResults(const float *cpu_result, const float *gpu_result,
                    size_t size, float atol = 1e-4f, float rtol = 1e-5f)
{
    for (size_t i = 0; i < size; i++)
    {
        float a = cpu_result[i];
        float b = gpu_result[i];
        float abs_diff = std::fabs(a - b);
        float rel_diff = abs_diff / std::fmax(std::fabs(a), std::fabs(b));

        if (abs_diff > atol && rel_diff > rtol)
        {
            std::cout << "Mismatch at index " << i
                      << ": CPU = " << a
                      << ", GPU = " << b
                      << ", abs diff = " << abs_diff
                      << ", rel diff = " << rel_diff
                      << std::endl;
            return false;
        }
    }
    return true;
}

#endif


Writing utils.h


In [18]:

%%writefile conv_2d_constant.cu
#include <cuda_bf16.h>
#include <cuda_fp16.h>
#include <cuda_fp8.h>
#include <cuda_runtime.h>
#include <float.h>
#include <stdio.h>
#include <stdlib.h>
#include <cmath>
#include <cstddef>
#include <iostream>
#include "utils.h"

#define FILTER_RADIUS 2
#define FILTER_H (2 * FILTER_RADIUS + 1)
#define FILTER_W (2 * FILTER_RADIUS + 1)
#define FILTER_ELEMENTS (FILTER_H * FILTER_W)
__constant__ float c_filter[FILTER_ELEMENTS];

/**
 * Assuming filter is square, thats why I only calculate radius once, otherwise I would need radius_y and radius_x
 */
__global__ void constant_conv_2d(const float* x, float* y, int M, int K) {
    int ty = blockIdx.y * blockDim.y + threadIdx.y;
    int tx = blockIdx.x * blockDim.x + threadIdx.x;

    if (ty >= M || tx >= K) return;

    float res = 0.0f;

    for (int i = 0; i < FILTER_H; i++){
        for (int j = 0; j < FILTER_W; j++) {
            int out_row = ty - FILTER_RADIUS + i;
            int out_col = tx - FILTER_RADIUS + j;
            float val = (out_row >= 0 && out_row < M && out_col >=0 && out_col < K) ? x[out_row * K + out_col] : 0.0f;
            res += val * c_filter[i * FILTER_W + j];
        }
    }

    y[ty * K + tx] = res;
}

// Naive CPU reference: 2D convolution with zero padding.
// Input  : in  (H x W) row-major
// Filter : filt (FH x FW) row-major (typically odd dims)
// Output : out (H x W) row-major
//
// out[y, x] = sum_{j=0..FH-1} sum_{i=0..FW-1}
//               in[y + (j-ry), x + (i-rx)] * filt[j, i]
// where ry = FH/2, rx = FW/2, and out-of-bounds in[] reads are treated as 0.
void conv2d_cpu_ref_zero_pad(const float* in,
                             const float* filt,
                             float* out,
                             int H, int W,
                             int FH, int FW)
{
    const int ry = FH / 2;
    const int rx = FW / 2;

    for (int y = 0; y < H; ++y) {
        for (int x = 0; x < W; ++x) {
            float acc = 0.0f;

            for (int j = 0; j < FH; ++j) {
                for (int i = 0; i < FW; ++i) {
                    const int in_y = y + (j - ry);
                    const int in_x = x + (i - rx);

                    const float v =
                        (in_y >= 0 && in_y < H && in_x >= 0 && in_x < W)
                            ? in[in_y * W + in_x]
                            : 0.0f;

                    acc += v * filt[j * FW + i];
                }
            }

            out[y * W + x] = acc;
        }
    }
}

int main() {
    int M = 1 << 10;
    int K = 1 << 10;
    int f_h = 5;
    int f_w = 5;

    // Calculate size of 1D vector
    size_t size_x = M * K * sizeof(float);
    // Calculate size of 1D convolution filter
    size_t size_f = f_h * f_w * sizeof(float);
    size_t size_y = M * K * sizeof(float);

    // Create host pointers and allocate host memory
    float* X_host = (float*)malloc(size_x);
    float* F_host = (float*)malloc(size_f);
    float* Y_host_cpu = (float*)malloc(size_y);
    float* Y_host_gpu = (float*)malloc(size_y);

    // Initialise the arrays 
    float* array_X[] {X_host};
    float* array_F[] {F_host};
    initialiseArrays(array_X, 1, M * K, 0.0f, 100.0f, 0);
    initialiseArrays(array_F, 1, f_h * f_w, 1.0f, 50.0f, 0);

    // Measure CPU reference time
    float cpu_time = measureExecutionTime([&](){
        conv2d_cpu_ref_zero_pad(X_host, F_host, Y_host_cpu, M, K, f_h, f_w);
    });

    // Create device ptrs and allocate device memory
    float* X_device;
    float* F_device; 
    float* Y_device;

    cudaMalloc((void**)&X_device, size_x);
    cudaMalloc((void**)&F_device, size_f);
    cudaMalloc((void**)&Y_device, size_y);

    // Copy arrays to device memory
    cudaMemcpy(X_device, X_host, size_x, cudaMemcpyHostToDevice);
    cudaMemcpy(F_device, F_host, size_f, cudaMemcpyHostToDevice);

    // Copy host filter to device constant memory
    cudaMemcpyToSymbol(c_filter, F_host, size_f);

    // Configure grid and block dims
    int num_blocks_row = (M + 32 - 1 ) / 32;
    int num_blocks_col = (K + 32 - 1 ) / 32;
    dim3 blockDim(32, 32);
    dim3 gridDim(num_blocks_row, num_blocks_col);

    // Execute kernel & measure time
    double gpu_time = measureKernelTime([&]() {
        constant_conv_2d<<<gridDim, blockDim>>>(X_device, Y_device, M, K);
        cudaDeviceSynchronize();
    });

    // Copy results back to host
    cudaMemcpy(Y_host_gpu, Y_device, size_y, cudaMemcpyDeviceToHost);

    // Compare against cpu reference for correctness
    bool result_match = compareResults(Y_host_cpu, Y_host_gpu, M * K, 1e-4, 1e-5);
    std::cout << (result_match ? "Results match!" : "Results do not match!") << std::endl;

    // Free memories
    free(X_host);
    free(F_host);
    free(Y_host_cpu);
    free(Y_host_gpu);

    cudaFree(X_device);
    cudaFree(F_device);
    cudaFree(Y_device);
    
    return 0;
}


Overwriting conv_2d_constant.cu


In [20]:
!nvcc -O2 -arch=sm_70 conv_2d_constant.cu -o conv_2d_constant
!./conv_2d_constant

Results match!
